# Baseline vs. авторски модел – прогнозиране на цена за нощувка (StayOra)

Тетрадка за Google Colab. Сравнява тривиален модел, класически модел и авторски модел по MAE и RMSE.
Авторският модел се различава от базовия само по един инженеринг признак: `stars * log1p(review_count)`.

## 1. Инсталация и импорти

In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
np.random.seed(42)


## 2. Генериране на синтетичен dataset (StayOra-подобен)

In [ ]:
N = 1200
cities = ['Sofia','Plovdiv','Varna','Bansko','Burgas','London','Berlin','Paris','Madrid','Vienna']
city_boost = {'Sofia':35,'Plovdiv':10,'Varna':25,'Bansko':40,'Burgas':20,'London':60,'Berlin':45,'Paris':70,'Madrid':50,'Vienna':55}
types = ['hotel','apartment','villa','resort']
type_boost = {'hotel':0,'apartment':-10,'villa':25,'resort':40}

city = np.random.choice(cities, size=N, p=[0.12,0.08,0.1,0.15,0.08,0.1,0.1,0.08,0.09,0.1])
ptype = np.random.choice(types, size=N, p=[0.45,0.3,0.15,0.1])
stars = np.random.choice([1,2,3,4,5], size=N, p=[0.05,0.1,0.25,0.4,0.2])
reviews = np.random.poisson(lam=80, size=N).clip(0,600)
capacity = np.random.choice([1,2,3,4,6], size=N, p=[0.1,0.35,0.3,0.2,0.05])

base = 40 + 28*stars + 0.12*reviews + capacity*8
price = (
    base
    + np.array([city_boost[c] for c in city])
    + np.array([type_boost[t] for t in ptype])
    + 0.05*stars*reviews          # нелинейно взаимодействие
    + np.random.normal(0, 12, size=N)
)
price = np.clip(price, 15, None)

df = pd.DataFrame({
    'city': city,
    'property_type': ptype,
    'stars': stars,
    'review_count': reviews,
    'capacity': capacity,
    'price_per_night': price.round(2),
})
print(df.shape); df.head()
df.to_csv('stayora_prices.csv', index=False)


### Описателна статистика

In [ ]:
df.describe()
try:
    pass
except Exception:
    pass
print(df['property_type'].value_counts())

## 3. Feature engineering (само едно добавено нещо за авторския модел)

In [ ]:
import numpy as np
df['engineered_stars_reviews'] = df['stars'] * np.log1p(df['review_count'])
features_base = ['stars', 'review_count']
features_auth = features_base + ['engineered_stars_reviews']
X = df
y = df['price_per_night'].values


## 4. Train / Test split (80/20, seed=42)

In [ ]:
Xb = df[features_base].values
Xa = df[features_auth].values
y = df['price_per_night'].values

Xb_train, Xb_test, ya_train, ya_test = train_test_split(Xb, y, test_size=0.2, random_state=42)
Xa_train, Xa_test, y_train, y_test = train_test_split(Xa, y, test_size=0.2, random_state=42)
assert np.allclose(ya_test, y_test)
print('Train:', Xa_train.shape, 'Test:', Xa_test.shape)


## 5. Модели

In [ ]:
# Тривиален модел
dummy = DummyRegressor(strategy='mean')
dummy.fit(Xb_train, ya_train)
pred_dummy = dummy.predict(Xb_test)

# Класически (RF, 2 признака)
rf_base = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_base.fit(Xb_train, ya_train)
pred_base = rf_base.predict(Xb_test)

# Авторски (RF + 1 engineered признак)
rf_auth = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_auth.fit(Xa_train, y_train)
pred_auth = rf_auth.predict(Xa_test)


## 6. Метрики (MAE, RMSE, R2)

In [ ]:
def metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name:22s} MAE={mae:7.3f}  RMSE={rmse:7.3f}  R2={r2:6.3f}')
    return {'model':name,'MAE':mae,'RMSE':rmse,'R2':r2}

res = []
res.append(metrics(y_test, pred_dummy, 'Dummy (mean)'))
res.append(metrics(y_test, pred_base, 'RF baseline (2 feat)'))
res.append(metrics(y_test, pred_auth, 'RF authored (+1 feat)'))
results_df = pd.DataFrame(res).set_index('model')
results_df


## 7. Намаление на грешката (vs. Dummy)

In [ ]:
mae_dummy = results_df.loc['Dummy (mean)','MAE']
mae_base = results_df.loc['RF baseline (2 feat)','MAE']
mae_auth = results_df.loc['RF authored (+1 feat)','MAE']
print(f'Relative MAE reduction Dummy->Authored: {(1-mae_auth/mae_dummy)*100:.2f}%')
print(f'Relative MAE reduction Baseline->Authored: {(1-mae_auth/mae_base)*100:.2f}%')
results_df.plot(kind='bar', y=['MAE','RMSE'], figsize=(8,4), title='Model comparison')
plt.ylabel('Error (EUR)'); plt.show()


## 8. 5-кратна кръстосана проверка и paired t-тест (H0/H1)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_dummy_cv, mae_base_cv, mae_auth_cv = [], [], []
for tr, te in kf.split(Xa):
    # Dummy
    d = DummyRegressor(strategy='mean'); d.fit(Xb[tr], y[tr]);
    mae_dummy_cv.append(mean_absolute_error(y[te], d.predict(Xb[te])))
    # RF base
    rb = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1); rb.fit(Xb[tr], y[tr]);
    mae_base_cv.append(mean_absolute_error(y[te], rb.predict(Xb[te])))
    # RF authored
    ra = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1); ra.fit(Xa[tr], y[tr]);
    mae_auth_cv.append(mean_absolute_error(y[te], ra.predict(Xa[te])))

t_stat, p_val = stats.ttest_rel(mae_dummy_cv, mae_auth_cv)
print('MAE CV Dummy :', np.round(mae_dummy_cv,3))
print('MAE CV RFauth:', np.round(mae_auth_cv,3))
print(f'Paired t-test Dummy vs Authored: t={t_stat:.3f}, p={p_val:.6f}')
print('H0 rejected (p<0.05):', p_val < 0.05)


## 9. Заключение

In [ ]:
print('Ако p < 0.05 и MAE_authored < MAE_dummy с >=5% относително подобрение -> H0 е опровергана, H1 се приема.')
print('Авторският модел се различава от базовия САМО по engineered_stars_reviews, което гарантира валиден експеримент (ceteris paribus).')
